# 03 — Baseline Models (Image-only, Text-only, Multimodal)

**Phase 6.** Three baseline families, all evaluated with the *same* 5-fold
split produced by notebook 02, so the numbers are directly comparable and can
go straight into a results table.

| Family | Input | Models |
|---|---|---|
| Image-only | ultrasound frames | ResNet50, DenseNet121, EfficientNetV2-S |
| Text-only | **findings section only** | TF-IDF + linear, ClinicalBERT/BioBERT |
| Multimodal | frames + findings | late fusion of the two best |

### The one thing that will sink your paper if you get it wrong

Disease labels in this project were extracted from the **Impression** section.
A text model that reads the impression therefore scores ~0.99 F1 and has
learned nothing — it is reading the answer key. Reviewers catch this
immediately. Every text model below is fed `findings_proc`, which notebook 02
built with the impression and advice stripped out. The assertion cell enforces
it; do not disable it.

Because labels come from the report, the *honest* framing of the image-only
task is: "can the image alone recover the finding the radiologist wrote down?"
For a 259-study dataset the answer will be "partly, for the common classes."
That is a publishable result when reported honestly with confidence intervals
(notebook 06) — it is not publishable when dressed up as 95% accuracy.

Outputs to `artifacts/`: `oof_<model>.npy` (out-of-fold probabilities),
`baseline_results.csv`, `baseline_per_class.csv`.

In [1]:
import os
import sys
import re
import json
import time
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

SEED = 42
np.random.seed(SEED)


def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


ART = Path("/kaggle/working/artifacts")
FIG = ART / "figures"
for d in (ART, FIG):
    d.mkdir(parents=True, exist_ok=True)


# Artifacts are resolved one file at a time, because on Kaggle each notebook's
# output is a separate dataset: master_split.csv comes from notebook 02 while
# label_tiers.json and ontology.json come from notebook 00. Attach whichever
# outputs you need — this searches all of them.
PRODUCED_BY = {
    "master.csv": "00", "ontology.json": "00", "label_tiers.json": "00",
    "label_audit.csv": "00", "labels_multihot.csv": "00 (refreshed by 02)",
    "master_split.csv": "02", "frames.csv": "02", "preprocess_config.json": "02",
}


def _rank(p):
    """Prefer the most downstream notebook when a file exists in several inputs."""
    nums = re.findall(r"[/\\](\d{2})[-_]", str(p))
    return max((int(n) for n in nums), default=-1)


def find_file(fname, required=True):
    if (ART / fname).exists():
        return ART / fname
    hits = sorted(Path("/kaggle/input").rglob(fname), key=_rank, reverse=True)
    if hits:
        return hits[0]
    if not required:
        return None
    raise FileNotFoundError(
        f"'{fname}' not found in /kaggle/working/artifacts or any attached dataset.\n"
        f"It is produced by notebook {PRODUCED_BY.get(fname, '00-02')}. "
        "Use Add Data -> Notebook Output to attach that notebook's output, "
        "then re-run this cell."
    )


for _f in ("master_split.csv", "frames.csv", "label_tiers.json", "labels_multihot.csv"):
    print(f"  {_f:24s} <- {find_file(_f).parent}")

df = pd.read_csv(find_file("master_split.csv"))
frames = pd.read_csv(find_file("frames.csv"))
tiers = json.loads(find_file("label_tiers.json").read_text())
Ydf = pd.read_csv(find_file("labels_multihot.csv"))

df["labels"] = df["labels"].fillna("").map(lambda s: [x for x in s.split("|") if x])
print(f"{len(df)} studies, {len(frames)} frames, folds: {sorted(df['fold'].unique())}")

  master_split.csv         <- /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/artifacts
  frames.csv               <- /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/artifacts
  label_tiers.json         <- /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/artifacts
  labels_multihot.csv      <- /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/artifacts
259 studies, 494 frames, folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


### Re-anchor frame paths

`frames.csv` stores the absolute paths that existed **while notebook 02 was
running** — `/kaggle/working/frames/...`. In this session those frames live
under `/kaggle/input/<notebook-02-output>/frames/` instead, so every path in
the file is stale.

This matters more than a normal path bug: `cv2.imread` returns `None` for a
missing file rather than raising, the loader substitutes a black image, and
training completes normally on blank inputs. You get plausible-looking loss
curves and meaningless results. The cell below re-anchors the paths and then
asserts the frames actually load.

In [2]:
def _extract_frames_archive():
    """Fall back to frames.zip when the loose frames/ directory is unavailable."""
    import zipfile
    hits = sorted(Path("/kaggle/input").rglob("frames.zip"))
    if not hits:
        return None
    dest = Path("/kaggle/working/frames")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(hits[0]) as zf:
        zf.extractall(dest)
        n = len(zf.namelist())
    print(f"Extracted {n} frames from {hits[0]} -> {dest}")
    return dest


def resolve_frames_root(frames_df):
    """Find the directory that actually holds the frames in this session."""
    sample = Path(str(frames_df["frame_path"].iloc[0]))
    if sample.exists():
        return None                                   # paths are already valid
    rel = Path(sample.parent.name) / sample.name      # e.g. R1/R1_p00.jpg
    candidates = [Path("/kaggle/working/frames")]
    for depth in ("*/frames", "*/*/frames", "*/frames/*"):
        candidates += sorted(Path("/kaggle/input").glob(depth))
    for c in candidates:
        if (c / rel).exists():
            return c
    unzipped = _extract_frames_archive()              # last resort: frames.zip
    if unzipped is not None and (unzipped / rel).exists():
        return unzipped
    raise FileNotFoundError(
        f"Frames not found. frames.csv points at {sample}, which does not exist "
        "here, and no attached dataset contains a 'frames/' folder or a "
        "'frames.zip' holding that file.\n"
        "Re-run notebook 02 (it now writes /kaggle/working/frames.zip), save the "
        "version, and attach that output here."
    )


_root = resolve_frames_root(frames)
if _root is not None:
    frames["frame_path"] = [
        str(_root / Path(p).parent.name / Path(p).name) for p in frames["frame_path"]]
    print(f"Re-anchored {len(frames)} frame paths to {_root}")
else:
    print("Frame paths already valid.")

_missing = [p for p in frames["frame_path"] if not os.path.exists(p)]
assert not _missing, (
    f"{len(_missing)} of {len(frames)} frame files are missing, e.g. {_missing[:3]}. "
    "Re-save notebook 02's output including the frames/ directory."
)

# Decode one frame and confirm it is not blank — a file can exist and still be
# unreadable, and a silently black dataset is the failure this guards against.
import cv2 as _cv2
_probe = _cv2.imread(frames["frame_path"].iloc[0])
assert _probe is not None, f"cv2 could not decode {frames['frame_path'].iloc[0]}"
assert _probe.std() > 1.0, (
    "First frame decodes to a near-constant image — the frames are blank. "
    "Re-run notebook 02's panel decomposition."
)
print(f"Frames verified: {len(frames)} files, sample {_probe.shape}, "
      f"pixel std {_probe.std():.1f}")

Extracted 494 frames from /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/frames.zip -> /kaggle/working/frames
Re-anchored 494 frame paths to /kaggle/working/frames
Frames verified: 494 files, sample (224, 224, 3), pixel std 99.3


## 1. Target definition

We model the **trainable** and **marginal** tiers as a multi-label problem.
Classes rarer than that cannot be cross-validated honestly — with 3 positives
a single fold may contain zero, and F1 becomes undefined. They stay in the
dataset (and in the report-generation task, which does not need class counts)
but they are excluded from the classification target and reported as a
"long tail not modelled" row in the paper.

In [3]:
TARGETS = tiers["trainable"] + tiers["marginal"]
TARGETS = [t for t in TARGETS if t in Ydf.columns]
Y = Ydf[TARGETS].values.astype(np.float32)
folds = df["fold"].values

support = Y.sum(0).astype(int)
tbl = pd.DataFrame({"label": TARGETS, "n_positive": support,
                    "prevalence": (support / len(Y)).round(3)})
tbl["min_per_fold"] = [int(min(Y[folds == f, j].sum() for f in sorted(set(folds))))
                       for j in range(len(TARGETS))]
print(tbl.to_string(index=False))
print(f"\n{len(TARGETS)} target labels | mean labels/study = {Y.sum(1).mean():.2f}")

assert tbl["min_per_fold"].min() >= 1, (
    "A target class has an empty fold — re-run notebook 02's stratification or "
    "demote that class to the rare tier."
)

                  label  n_positive  prevalence  min_per_fold
           Normal study          84       0.324            16
       Normal pregnancy          42       0.162             8
            Fatty liver          28       0.108             5
               Cystitis          21       0.081             4
           Bulky uterus          16       0.062             3
           Hepatomegaly          15       0.058             3
Early pregnancy failure          14       0.054             2
        Significant PVR          10       0.039             2
      Pelvic collection           7       0.027             1
             Renal cyst           6       0.023             1
  Thickened endometrium           6       0.023             1
         Cholelithiasis           5       0.019             1
           Ovarian cyst           5       0.019             1

13 target labels | mean labels/study = 1.00


## 2. Metrics

Multi-label, so accuracy is meaningless; report:

- **micro-F1** — dominated by common classes, the headline number
- **macro-F1** — treats rare classes equally, the honest number
- **macro AUC** — threshold-free, robust to class imbalance
- **subset accuracy** — the strict "got every label right" rate

Thresholds are tuned **per class on the training folds only** and applied to
the held-out fold. Tuning on the test fold is the second most common leak in
student multi-label papers.

In [4]:
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score,
                             accuracy_score)


def tune_thresholds(y_true, y_prob, grid=np.arange(0.05, 0.95, 0.025)):
    """Per-class threshold maximising F1 on the given (training) data."""
    th = np.full(y_true.shape[1], 0.5)
    for j in range(y_true.shape[1]):
        if y_true[:, j].sum() == 0:
            continue
        best, best_t = -1.0, 0.5
        for t in grid:
            f = f1_score(y_true[:, j], (y_prob[:, j] >= t).astype(int), zero_division=0)
            if f > best:
                best, best_t = f, t
        th[j] = best_t
    return th


def safe_auc(y_true, y_prob):
    aucs = []
    for j in range(y_true.shape[1]):
        if 0 < y_true[:, j].sum() < len(y_true):
            aucs.append(roc_auc_score(y_true[:, j], y_prob[:, j]))
    return float(np.mean(aucs)) if aucs else np.nan


def evaluate(y_true, y_prob, th, name=""):
    y_pred = (y_prob >= th).astype(int)
    return {
        "model": name,
        "micro_F1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_P": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_R": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_AUC": safe_auc(y_true, y_prob),
        "macro_AP": float(np.mean([
            average_precision_score(y_true[:, j], y_prob[:, j])
            for j in range(y_true.shape[1]) if y_true[:, j].sum() > 0])),
        "subset_acc": accuracy_score(y_true, y_pred),
    }


def per_class_report(y_true, y_prob, th, names, model):
    rows = []
    y_pred = (y_prob >= th).astype(int)
    for j, n in enumerate(names):
        pos = int(y_true[:, j].sum())
        rows.append({
            "model": model, "label": n, "n_positive": pos,
            "precision": precision_score(y_true[:, j], y_pred[:, j], zero_division=0),
            "recall": recall_score(y_true[:, j], y_pred[:, j], zero_division=0),
            "f1": f1_score(y_true[:, j], y_pred[:, j], zero_division=0),
            "auc": (roc_auc_score(y_true[:, j], y_prob[:, j])
                    if 0 < pos < len(y_true) else np.nan),
        })
    return pd.DataFrame(rows)


RESULTS, PERCLASS, OOF = [], [], {}


def register(name, oof_prob):
    """Threshold-tune out-of-fold, score, and stash for notebook 06."""
    th_all = np.zeros((len(set(folds)), len(TARGETS)))
    y_pred = np.zeros_like(oof_prob)
    for i, f in enumerate(sorted(set(folds))):
        tr, te = folds != f, folds == f
        th = tune_thresholds(Y[tr], oof_prob[tr])
        th_all[i] = th
        y_pred[te] = (oof_prob[te] >= th).astype(float)
    th_mean = th_all.mean(0)
    res = evaluate(Y, oof_prob, th_mean, name)
    RESULTS.append(res)
    PERCLASS.append(per_class_report(Y, oof_prob, th_mean, TARGETS, name))
    OOF[name] = oof_prob
    np.save(ART / f"oof_{name}.npy", oof_prob)
    np.save(ART / f"thresholds_{name}.npy", th_mean)
    print(f"  {name:22s} microF1={res['micro_F1']:.3f}  macroF1={res['macro_F1']:.3f}  "
          f"AUC={res['macro_AUC']:.3f}")
    return res

## 3. Trivial baselines

Always publish these. A "predict the most frequent label set" baseline on an
imbalanced 259-study dataset can reach a micro-F1 that looks respectable, and
a reviewer who computes it will ask why your deep model is only 4 points
better. Knowing the floor lets you argue about the gap honestly.

In [5]:
# Prior baseline: predict each class at its training-fold prevalence.
oof_prior = np.zeros_like(Y)
for f in sorted(set(folds)):
    tr, te = folds != f, folds == f
    oof_prior[te] = Y[tr].mean(0)
register("prior", oof_prior)

# Majority baseline: always predict the single most common label.
oof_major = np.zeros_like(Y)
for f in sorted(set(folds)):
    tr, te = folds != f, folds == f
    j = Y[tr].sum(0).argmax()
    oof_major[te, j] = 1.0
register("majority", oof_major)

  prior                  microF1=0.212  macroF1=0.111  AUC=0.472
  majority               microF1=0.324  macroF1=0.038  AUC=0.500


{'model': 'majority',
 'micro_F1': 0.32432432432432434,
 'macro_F1': 0.03767660910518053,
 'micro_P': 0.32432432432432434,
 'micro_R': 0.32432432432432434,
 'macro_AUC': 0.5,
 'macro_AP': 0.0769230769230769,
 'subset_acc': 0.32432432432432434}

## 4. Text-only baselines

### 4a. TF-IDF + one-vs-rest logistic regression

Fast, deterministic, no GPU. On structured radiology prose this is a
surprisingly strong baseline and often beats a fine-tuned BERT on a few
hundred examples — a finding worth reporting rather than hiding.

In [6]:
TEXT_COL = "findings_proc"
assert TEXT_COL in df.columns, "Run notebook 02 — it creates findings_proc."

# Structural leakage guard: the impression block itself must be gone.
import re as _re
hdr = _re.compile(r"\b(IMPRESSION|COMMENTS?)\s*[:\-]", _re.I)
bad = [s for s, t in zip(df["study_id"], df[TEXT_COL].fillna("")) if hdr.search(t)]
assert not bad, (
    f"{len(bad)} rows still contain an Impression/Comment header inside "
    f"'{TEXT_COL}' (e.g. {bad[:3]}). Fix notebook 02's section splitter first."
)
print(f"Structural check passed: no impression block inside '{TEXT_COL}'.")

# Informational: how often does the impression phrase *also* appear verbatim in
# the findings? This is not a bug — radiologists restate the key finding — but
# it caps what a text-only result means, and you must say so in the paper.
echo = 0
for fnd, imp in zip(df[TEXT_COL].fillna(""), df["impression"].fillna("")):
    key = _re.sub(r"^(suggestive of|features of)\s+", "", imp.strip().lower())[:30]
    if len(key) > 12 and key in fnd.lower():
        echo += 1
print(f"Impression phrase restated verbatim in findings: {echo}/{len(df)} "
      f"({echo/len(df):.0%}) studies.")
print("  -> Text-only numbers are partly keyword matching against the radiologist's\n"
      "     own wording. Report them as an upper reference, not as a clinical model.")

texts = df[TEXT_COL].fillna("").values
print("mean findings length:", int(np.mean([len(t.split()) for t in texts])), "words")

Structural check passed: no impression block inside 'findings_proc'.
Impression phrase restated verbatim in findings: 2/259 (1%) studies.
  -> Text-only numbers are partly keyword matching against the radiologist's
     own wording. Report them as an upper reference, not as a clinical model.
mean findings length: 143 words


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

t0 = time.time()
oof_tfidf = np.zeros_like(Y)
for f in sorted(set(folds)):
    tr, te = folds != f, folds == f
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                          strip_accents="unicode")
    Xtr = vec.fit_transform(texts[tr])
    Xte = vec.transform(texts[te])
    for j in range(len(TARGETS)):
        if Y[tr, j].sum() < 2:
            oof_tfidf[te, j] = Y[tr, j].mean()
            continue
        clf = LogisticRegression(C=4.0, max_iter=2000, class_weight="balanced")
        clf.fit(Xtr, Y[tr, j])
        oof_tfidf[te, j] = clf.predict_proba(Xte)[:, 1]
register("tfidf_logreg", oof_tfidf)
print(f"  ({time.time() - t0:.1f}s)")

  tfidf_logreg           microF1=0.739  macroF1=0.680  AUC=0.923
  (5.5s)


In [8]:
# Linear SVM variant — often a point or two better on sparse text.
t0 = time.time()
oof_svm = np.zeros_like(Y)
for f in sorted(set(folds)):
    tr, te = folds != f, folds == f
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                          strip_accents="unicode")
    Xtr, Xte = vec.fit_transform(texts[tr]), vec.transform(texts[te])
    for j in range(len(TARGETS)):
        if Y[tr, j].sum() < 3:
            oof_svm[te, j] = Y[tr, j].mean()
            continue
        base = LinearSVC(C=0.5, class_weight="balanced")
        clf = CalibratedClassifierCV(base, cv=3, method="sigmoid")
        clf.fit(Xtr, Y[tr, j])
        oof_svm[te, j] = clf.predict_proba(Xte)[:, 1]
register("tfidf_svm", oof_svm)
print(f"  ({time.time() - t0:.1f}s)")

  tfidf_svm              microF1=0.733  macroF1=0.628  AUC=0.890
  (6.8s)


### 4b. ClinicalBERT / BioBERT fine-tuning

Needs a GPU and internet on (Settings → Internet → On) to pull weights from
the Hugging Face Hub. Set `RUN_BERT = False` to skip and still get a complete
results table from the linear models.

Checkpoints worth comparing in the paper:
`emilyalsentzer/Bio_ClinicalBERT`, `dmis-lab/biobert-base-cased-v1.2`,
`microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`.

In [9]:
RUN_BERT = True
BERT_CKPT = "emilyalsentzer/Bio_ClinicalBERT"
BERT_EPOCHS = 12
BERT_LR = 2e-5
BERT_MAXLEN = 320

try:
    import torch
    HAS_TORCH = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    HAS_TORCH = False
    DEVICE = "cpu"
print("torch:", HAS_TORCH, "| device:", DEVICE)

if RUN_BERT and HAS_TORCH:
    try:
        import transformers  # noqa
    except ImportError:
        pipq("transformers")
    try:
        from transformers import AutoTokenizer, AutoModel
        HAS_HF = True
    except ImportError:
        HAS_HF = False
        print("transformers unavailable — skipping BERT baseline.")
else:
    HAS_HF = False

torch: True | device: cuda


In [10]:
if RUN_BERT and HAS_HF:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader

    class TextDS(Dataset):
        def __init__(self, texts, y, tok):
            self.enc = tok(list(texts), truncation=True, max_length=BERT_MAXLEN,
                           padding="max_length", return_tensors="pt")
            self.y = torch.tensor(y)

        def __len__(self):
            return len(self.y)

        def __getitem__(self, i):
            return ({k: v[i] for k, v in self.enc.items()}, self.y[i])

    class BertMultiLabel(nn.Module):
        def __init__(self, ckpt, n_out):
            super().__init__()
            self.enc = AutoModel.from_pretrained(ckpt)
            h = self.enc.config.hidden_size
            self.drop = nn.Dropout(0.2)
            self.head = nn.Linear(h, n_out)

        def forward(self, batch):
            out = self.enc(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1).float()
            pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
            return self.head(self.drop(pooled))

    def run_bert(ckpt, tag):
        tok = AutoTokenizer.from_pretrained(ckpt)
        oof = np.zeros_like(Y)
        pos_w = torch.tensor(((len(Y) - Y.sum(0)) / np.maximum(Y.sum(0), 1)).clip(1, 20),
                             dtype=torch.float32, device=DEVICE)
        for f in sorted(set(folds)):
            tr, te = folds != f, folds == f
            model = BertMultiLabel(ckpt, len(TARGETS)).to(DEVICE)
            opt = torch.optim.AdamW(model.parameters(), lr=BERT_LR, weight_decay=0.01)
            dl = DataLoader(TextDS(texts[tr], Y[tr], tok), batch_size=8, shuffle=True)
            sched = torch.optim.lr_scheduler.OneCycleLR(
                opt, max_lr=BERT_LR, total_steps=BERT_EPOCHS * len(dl))
            lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w)
            model.train()
            for ep in range(BERT_EPOCHS):
                for xb, yb in dl:
                    xb = {k: v.to(DEVICE) for k, v in xb.items()}
                    loss = lossf(model(xb), yb.to(DEVICE))
                    opt.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                    sched.step()
            model.eval()
            probs = []
            with torch.no_grad():
                for xb, _ in DataLoader(TextDS(texts[te], Y[te], tok), batch_size=16):
                    xb = {k: v.to(DEVICE) for k, v in xb.items()}
                    probs.append(torch.sigmoid(model(xb)).cpu().numpy())
            oof[te] = np.concatenate(probs)
            del model
            torch.cuda.empty_cache()
            print(f"    fold {f} done")
        register(tag, oof)
        return oof

    t0 = time.time()
    oof_bert = run_bert(BERT_CKPT, "clinicalbert")
    print(f"  ({time.time() - t0:.0f}s total)")
else:
    oof_bert = None

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    fold 3 done


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    fold 4 done
  clinicalbert           microF1=0.456  macroF1=0.456  AUC=0.853
  (775s total)


## 5. Image-only baselines

### Multiple-instance learning, because that is what this data actually is

Each study is a sheet of 2–8 ultrasound frames and the label belongs to the
**study**, not to any one frame. A "Cholelithiasis" study has a gallbladder
frame showing the stone and several unrelated frames. Training each frame
with the study label teaches the network that normal-looking kidneys mean
gallstones.

So: encode every frame, then pool the frame representations into one study
representation with attention-based MIL (Ilse et al., 2018). The attention
weights are also free explainability — notebook 06 shows which frame drove
each prediction, which is exactly the "which region influenced the decision"
claim in the proposal.

In [11]:
IMG_SIZE = 224
BATCH_STUDIES = 8          # studies per batch (each carries several frames)
MAX_FRAMES = 8             # cap frames per study for memory
EPOCHS = 25
LR_BACKBONE = 1e-4
LR_HEAD = 1e-3
RUN_IMAGE = True

if RUN_IMAGE and HAS_TORCH:
    try:
        import timm  # noqa
        HAS_TIMM = True
    except ImportError:
        pipq("timm")
        try:
            import timm  # noqa
            HAS_TIMM = True
        except ImportError:
            HAS_TIMM = False
else:
    HAS_TIMM = False
print("timm:", HAS_TIMM)

# Map study -> its frame paths, dropping studies whose frames went missing.
frames_by_study = frames.groupby("study_id")["frame_path"].apply(list).to_dict()
missing = [s for s in df["study_id"] if s not in frames_by_study]
if missing:
    print(f"WARNING: {len(missing)} studies have no frames: {missing[:5]}")
study_ids = df["study_id"].values

timm: True


In [12]:
if RUN_IMAGE and HAS_TORCH and HAS_TIMM:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    import cv2

    torch.manual_seed(SEED)

    MEAN = np.array([0.485, 0.456, 0.406], np.float32)
    STD = np.array([0.229, 0.224, 0.225], np.float32)

    def load_frame(path, train):
        img = cv2.imread(str(path))
        if img is None:
            # Never substitute a blank image here: the model would train
            # happily on zeros and report meaningless metrics. Paths were
            # verified above, so a failure now is a genuinely corrupt file.
            raise FileNotFoundError(f"cv2 could not read frame: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if img.shape[:2] != (IMG_SIZE, IMG_SIZE):
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        img = img.astype(np.float32) / 255.0
        if train:
            # Ultrasound-appropriate augmentation only.
            # No horizontal flip: laterality (right vs left kidney/ovary) is
            # clinically meaningful and appears in the reports.
            if np.random.rand() < 0.5:
                ang = np.random.uniform(-10, 10)
                M = cv2.getRotationMatrix2D((IMG_SIZE / 2, IMG_SIZE / 2), ang, 1.0)
                img = cv2.warpAffine(img, M, (IMG_SIZE, IMG_SIZE),
                                     borderMode=cv2.BORDER_CONSTANT, borderValue=0)
            if np.random.rand() < 0.5:
                img = np.clip(img * np.random.uniform(0.85, 1.15)
                              + np.random.uniform(-0.06, 0.06), 0, 1)
            if np.random.rand() < 0.3:      # gain/speckle jitter
                img = np.clip(img + np.random.normal(0, 0.02, img.shape).astype(np.float32), 0, 1)
            if np.random.rand() < 0.3:
                s = np.random.uniform(0.85, 1.0)
                h = int(IMG_SIZE * s)
                y0 = np.random.randint(0, IMG_SIZE - h + 1)
                x0 = np.random.randint(0, IMG_SIZE - h + 1)
                img = cv2.resize(img[y0:y0 + h, x0:x0 + h], (IMG_SIZE, IMG_SIZE))
        return (img - MEAN) / STD

    class StudyDS(Dataset):
        """One item = one study = a bag of frames."""

        def __init__(self, sids, y, train):
            self.sids, self.y, self.train = sids, y, train

        def __len__(self):
            return len(self.sids)

        def __getitem__(self, i):
            paths = frames_by_study.get(self.sids[i], [])
            if not paths:
                paths = []
            if len(paths) > MAX_FRAMES:
                idx = (np.random.choice(len(paths), MAX_FRAMES, replace=False)
                       if self.train else np.linspace(0, len(paths) - 1, MAX_FRAMES).astype(int))
                paths = [paths[k] for k in sorted(idx)]
            imgs = [load_frame(p, self.train) for p in paths] or \
                   [np.zeros((IMG_SIZE, IMG_SIZE, 3), np.float32)]
            x = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float()
            return x, torch.tensor(self.y[i]), len(imgs)

    def collate(batch):
        """Pad variable-size bags and return a validity mask."""
        n = max(b[2] for b in batch)
        xs, ys, ms = [], [], []
        for x, y, k in batch:
            pad = n - k
            if pad:
                x = torch.cat([x, torch.zeros(pad, *x.shape[1:])], 0)
            m = torch.zeros(n)
            m[:k] = 1
            xs.append(x)
            ys.append(y)
            ms.append(m)
        return torch.stack(xs), torch.stack(ys), torch.stack(ms)

    class AttentionMIL(nn.Module):
        """Frame encoder + gated attention pooling + multi-label head."""

        def __init__(self, backbone, n_out, pretrained=True, att_dim=128):
            super().__init__()
            self.enc = timm.create_model(backbone, pretrained=pretrained,
                                         num_classes=0, global_pool="avg")
            d = self.enc.num_features
            self.att_V = nn.Sequential(nn.Linear(d, att_dim), nn.Tanh())
            self.att_U = nn.Sequential(nn.Linear(d, att_dim), nn.Sigmoid())
            self.att_w = nn.Linear(att_dim, 1)
            self.drop = nn.Dropout(0.3)
            self.head = nn.Linear(d, n_out)

        def forward(self, x, mask, return_att=False):
            B, N = x.shape[:2]
            h = self.enc(x.flatten(0, 1)).view(B, N, -1)          # B,N,d
            a = self.att_w(self.att_V(h) * self.att_U(h)).squeeze(-1)   # B,N
            a = a.masked_fill(mask == 0, -1e4)
            a = torch.softmax(a, dim=1)
            z = (a.unsqueeze(-1) * h).sum(1)                       # B,d
            logits = self.head(self.drop(z))
            return (logits, a) if return_att else logits

    def train_image_model(backbone, tag, epochs=EPOCHS):
        pos_w = torch.tensor(((len(Y) - Y.sum(0)) / np.maximum(Y.sum(0), 1)).clip(1, 15),
                             dtype=torch.float32, device=DEVICE)
        oof = np.zeros_like(Y)
        att_store = {}
        for f in sorted(set(folds)):
            tr, te = folds != f, folds == f
            model = AttentionMIL(backbone, len(TARGETS)).to(DEVICE)
            head_p = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
            enc_p = [p for n, p in model.named_parameters() if n.startswith("enc.")]
            opt = torch.optim.AdamW(
                [{"params": enc_p, "lr": LR_BACKBONE},
                 {"params": head_p, "lr": LR_HEAD}], weight_decay=1e-4)
            dl = DataLoader(StudyDS(study_ids[tr], Y[tr], True),
                            batch_size=BATCH_STUDIES, shuffle=True,
                            collate_fn=collate, num_workers=2, drop_last=False)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(dl))
            lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w)
            scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
            model.train()
            for ep in range(epochs):
                tot = 0.0
                for xb, yb, mb in dl:
                    xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
                    with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                        loss = lossf(model(xb, mb), yb)
                    opt.zero_grad()
                    scaler.scale(loss).backward()
                    scaler.step(opt)
                    scaler.update()
                    sched.step()
                    tot += loss.item()
                if (ep + 1) % 10 == 0:
                    print(f"    fold {f} ep {ep+1:3d}  loss {tot/len(dl):.4f}")
            model.eval()
            probs, atts = [], []
            with torch.no_grad():
                for xb, yb, mb in DataLoader(StudyDS(study_ids[te], Y[te], False),
                                             batch_size=8, collate_fn=collate):
                    lg, a = model(xb.to(DEVICE), mb.to(DEVICE), return_att=True)
                    probs.append(torch.sigmoid(lg).cpu().numpy())
                    atts.append(a.cpu().numpy())
            oof[te] = np.concatenate(probs)
            for sid, a in zip(study_ids[te], np.concatenate(
                    [np.pad(a, ((0, 0), (0, MAX_FRAMES - a.shape[1])), constant_values=0)
                     if a.shape[1] < MAX_FRAMES else a[:, :MAX_FRAMES] for a in atts])):
                att_store[sid] = a
            torch.save(model.state_dict(), ART / f"{tag}_fold{f}.pt")
            del model
            torch.cuda.empty_cache()
        np.save(ART / f"attention_{tag}.npy", att_store, allow_pickle=True)
        register(tag, oof)
        return oof

    IMAGE_BACKBONES = [
        ("resnet50", "img_resnet50"),
        ("densenet121", "img_densenet121"),
        ("tf_efficientnetv2_s", "img_effnetv2s"),
    ]
    image_oofs = {}
    for bb, tag in IMAGE_BACKBONES:
        print(f"\n=== {bb} ===")
        t0 = time.time()
        try:
            image_oofs[tag] = train_image_model(bb, tag)
            print(f"  ({(time.time()-t0)/60:.1f} min)")
        except Exception as e:                      # keep the notebook alive
            print(f"  FAILED: {type(e).__name__}: {e}")
else:
    image_oofs = {}
    print("Image baselines skipped (torch/timm unavailable).")


=== resnet50 ===


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

    fold 0 ep  10  loss 0.9462
    fold 0 ep  20  loss 0.9006
    fold 1 ep  10  loss 0.9649
    fold 1 ep  20  loss 0.7760
    fold 2 ep  10  loss 0.9342
    fold 2 ep  20  loss 0.7708
    fold 3 ep  10  loss 0.9454
    fold 3 ep  20  loss 0.7822
    fold 4 ep  10  loss 0.9445
    fold 4 ep  20  loss 0.7942
  img_resnet50           microF1=0.286  macroF1=0.255  AUC=0.637
  (10.7 min)

=== densenet121 ===


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

    fold 0 ep  10  loss 0.5325
    fold 0 ep  20  loss 0.3188
    fold 1 ep  10  loss 0.4728
    fold 1 ep  20  loss 0.3142
    fold 2 ep  10  loss 0.5932
    fold 2 ep  20  loss 0.2693
    fold 3 ep  10  loss 0.5492
    fold 3 ep  20  loss 0.2901
    fold 4 ep  10  loss 0.5954
    fold 4 ep  20  loss 0.3251
  img_densenet121        microF1=0.390  macroF1=0.306  AUC=0.694
  (12.5 min)

=== tf_efficientnetv2_s ===


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

    fold 0 ep  10  loss 0.7362
    fold 0 ep  20  loss 0.2614
    fold 1 ep  10  loss 0.5406
    fold 1 ep  20  loss 0.2383
    fold 2 ep  10  loss 0.5238
    fold 2 ep  20  loss 0.2259
    fold 3 ep  10  loss 0.5944
    fold 3 ep  20  loss 0.2185
    fold 4 ep  10  loss 0.6391
    fold 4 ep  20  loss 0.2467
  img_effnetv2s          microF1=0.411  macroF1=0.315  AUC=0.700
  (15.7 min)


## 6. Multimodal baseline (late fusion)

The proposal calls for concatenating image and text embeddings. On 259
studies an early-fusion MLP overfits before it learns anything, so the
defensible baseline is **late fusion**: a per-class convex combination of the
two modalities' probabilities, with the weight chosen on training folds only.

Report both. "Early fusion overfit, late fusion worked" is a legitimate
finding on a small dataset and pre-empts the obvious reviewer question.

In [13]:
best_text = max([r for r in RESULTS if r["model"].startswith(("tfidf", "clinical"))],
                key=lambda r: r["macro_F1"], default=None)
best_img = max([r for r in RESULTS if r["model"].startswith("img_")],
               key=lambda r: r["macro_F1"], default=None)

if best_text and best_img:
    Pt, Pi = OOF[best_text["model"]], OOF[best_img["model"]]
    oof_fuse = np.zeros_like(Y)
    weights = np.zeros((len(set(folds)), len(TARGETS)))
    for i, f in enumerate(sorted(set(folds))):
        tr, te = folds != f, folds == f
        for j in range(len(TARGETS)):
            best_w, best_f1 = 0.5, -1
            for w in np.linspace(0, 1, 21):
                p = w * Pt[tr, j] + (1 - w) * Pi[tr, j]
                t = tune_thresholds(Y[tr][:, [j]], p.reshape(-1, 1))[0]
                s = f1_score(Y[tr, j], (p >= t).astype(int), zero_division=0)
                if s > best_f1:
                    best_f1, best_w = s, w
            weights[i, j] = best_w
            oof_fuse[te, j] = best_w * Pt[te, j] + (1 - best_w) * Pi[te, j]
    register("fusion_late", oof_fuse)
    np.save(ART / "fusion_weights.npy", weights)
    wdf = pd.DataFrame({"label": TARGETS, "mean_text_weight": weights.mean(0).round(3)})
    print("\nLearned modality weights (1.0 = text only, 0.0 = image only):")
    print(wdf.to_string(index=False))
else:
    print("Need one text and one image model before fusion.")

  fusion_late            microF1=0.676  macroF1=0.671  AUC=0.906

Learned modality weights (1.0 = text only, 0.0 = image only):
                  label  mean_text_weight
           Normal study              0.93
       Normal pregnancy              0.88
            Fatty liver              0.96
               Cystitis              0.79
           Bulky uterus              0.86
           Hepatomegaly              0.87
Early pregnancy failure              0.86
        Significant PVR              0.75
      Pelvic collection              0.96
             Renal cyst              0.90
  Thickened endometrium              0.81
         Cholelithiasis              0.82
           Ovarian cyst              0.38


## 7. Results

The table below is Table 2 of your paper. Notebook 06 adds bootstrap
confidence intervals and the McNemar tests that turn these point estimates
into defensible claims.

In [14]:
res_df = pd.DataFrame(RESULTS).sort_values("macro_F1", ascending=False)
num = res_df.select_dtypes(float).columns
res_df[num] = res_df[num].round(4)
res_df.to_csv(ART / "baseline_results.csv", index=False)
print(res_df.to_string(index=False))

pc = pd.concat(PERCLASS, ignore_index=True)
pc.to_csv(ART / "baseline_per_class.csv", index=False)

          model  micro_F1  macro_F1  micro_P  micro_R  macro_AUC  macro_AP  subset_acc
   tfidf_logreg    0.7385    0.6799   0.6808   0.8069     0.9230    0.6636      0.5714
    fusion_late    0.6765    0.6712   0.5864   0.7992     0.9059    0.6609      0.4556
      tfidf_svm    0.7327    0.6285   0.7101   0.7568     0.8896    0.6452      0.5792
   clinicalbert    0.4564    0.4556   0.3246   0.7683     0.8530    0.4733      0.1853
  img_effnetv2s    0.4110    0.3149   0.3185   0.5792     0.7003    0.2674      0.2471
img_densenet121    0.3901    0.3060   0.2868   0.6100     0.6944    0.2654      0.1699
   img_resnet50    0.2864    0.2547   0.1791   0.7143     0.6375    0.1789      0.0309
          prior    0.2124    0.1110   0.1213   0.8494     0.4718    0.0744      0.0000
       majority    0.3243    0.0377   0.3243   0.3243     0.5000    0.0769      0.3243


In [15]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
r = res_df.set_index("model")
r[["micro_F1", "macro_F1"]].plot.barh(ax=ax[0], width=0.78)
ax[0].set_title("Baseline comparison (5-fold out-of-fold)")
ax[0].set_xlabel("F1")
ax[0].invert_yaxis()
ax[0].grid(axis="x", alpha=0.3)

top = res_df.iloc[0]["model"]
sub = pc[pc["model"] == top].sort_values("f1")
ax[1].barh(sub["label"], sub["f1"], color="#4c72b0")
for i, (_, row) in enumerate(sub.iterrows()):
    ax[1].text(0.01, i, f" n={int(row['n_positive'])}", va="center", fontsize=8, color="white")
ax[1].set_title(f"Per-class F1 — {top}")
ax[1].set_xlabel("F1")
ax[1].grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / "baseline_comparison.png")
plt.show()

## 8. Where the errors are

Per-class F1 against class size tells you whether the model learned anything
beyond frequency. If F1 tracks support almost perfectly, the model is
predicting the prior and your image encoder is contributing nothing — an
important thing to discover now rather than after writing the discussion.

In [16]:
sub = pc[pc["model"] == top]
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(sub["n_positive"], sub["f1"], s=55, color="#c44e52")
for _, row in sub.iterrows():
    ax.annotate(row["label"], (row["n_positive"], row["f1"]), fontsize=7,
                xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("training positives")
ax.set_ylabel("F1")
ax.set_title(f"Does performance just track class size?  ({top})")
ax.grid(alpha=0.3)
corr = sub[["n_positive", "f1"]].corr().iloc[0, 1]
ax.text(0.98, 0.04, f"Spearman-ish r = {corr:.2f}", transform=ax.transAxes,
        ha="right", fontsize=9)
plt.savefig(FIG / "f1_vs_support.png")
plt.show()

In [17]:
summary = {
    "n_studies": int(len(df)),
    "n_frames": int(len(frames)),
    "n_targets": len(TARGETS),
    "targets": TARGETS,
    "best_model": top,
    "best_macro_F1": float(res_df.iloc[0]["macro_F1"]),
    "best_micro_F1": float(res_df.iloc[0]["micro_F1"]),
    "models_run": res_df["model"].tolist(),
}
(ART / "baseline_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "n_studies": 259,
  "n_frames": 494,
  "n_targets": 13,
  "targets": [
    "Normal study",
    "Normal pregnancy",
    "Fatty liver",
    "Cystitis",
    "Bulky uterus",
    "Hepatomegaly",
    "Early pregnancy failure",
    "Significant PVR",
    "Pelvic collection",
    "Renal cyst",
    "Thickened endometrium",
    "Cholelithiasis",
    "Ovarian cyst"
  ],
  "best_model": "tfidf_logreg",
  "best_macro_F1": 0.6799,
  "best_micro_F1": 0.7385,
  "models_run": [
    "tfidf_logreg",
    "fusion_late",
    "tfidf_svm",
    "clinicalbert",
    "img_effnetv2s",
    "img_densenet121",
    "img_resnet50",
    "prior",
    "majority"
  ]
}


## What to carry into the paper

1. **Table 2** = `baseline_results.csv`, with the prior/majority rows kept in.
2. The modality-weight table is a genuine finding: it shows *per class*
   whether the image or the report carries the signal. Classes where the
   weight sits near 1.0 are ones the image cannot see — say so explicitly in
   the discussion, because it is the honest limitation of ultrasound stills.
3. Out-of-fold probability matrices are saved. Notebook 06 needs them for
   McNemar tests; never recompute predictions for significance testing, use
   these.

Next: **04 — BiomedCLIP alignment and multi-task learning.**